In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
project_root = Path.cwd().parent.parent
raw_file = project_root / "data" / "raw" / "online_retail_II.xlsx"

print(raw_file)
print("File exists:", raw_file.exists())

/Users/nithinsaijatoth/Desktop/UK_Retail_Sales_Customer_Analytics/data/raw/online_retail_II.xlsx
File exists: True


In [3]:
df_2009_2010 = pd.read_excel(
    raw_file,
    sheet_name="Year 2009-2010"
)

df_2010_2011 = pd.read_excel(
    raw_file,
    sheet_name="Year 2010-2011"
)

df_2009_2010["SourceSheet"] = "Year 2009-2010"
df_2010_2011["SourceSheet"] = "Year 2010-2011"

In [4]:
df = pd.concat(
    [df_2009_2010, df_2010_2011],
    ignore_index=True
)

print("Starting rows:", len(df))
print("Starting columns:", len(df.columns))

df.head()

Starting rows: 1067371
Starting columns: 9


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010


In [5]:
before = len(df)

df = df.drop_duplicates().copy()

after = len(df)

print("Rows before:", before)
print("Rows after:", after)
print("Duplicates removed:", before - after)

Rows before: 1067371
Rows after: 1055238
Duplicates removed: 12133


In [6]:
df["IsCancelled"] = df["Invoice"].astype(str).str.startswith("C", na=False)

print(df["IsCancelled"].value_counts())

IsCancelled
False    1035805
True       19433
Name: count, dtype: int64


In [7]:
cancelled_df = df[df["IsCancelled"]].copy()
sales_df = df[~df["IsCancelled"]].copy()

print("Cancelled rows:", len(cancelled_df))
print("Non-cancelled rows:", len(sales_df))

Cancelled rows: 19433
Non-cancelled rows: 1035805


In [8]:
negative_sales_qty = sales_df[sales_df["Quantity"] < 0]

print("Negative quantity rows in non-cancelled sales:", len(negative_sales_qty))

negative_sales_qty.head()

Negative quantity rows in non-cancelled sales: 3457


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,IsCancelled
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom,Year 2009-2010,False
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom,Year 2009-2010,False
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom,Year 2009-2010,False
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom,Year 2009-2010,False
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom,Year 2009-2010,False


In [10]:
negative_sales_qty[
    ["Invoice", "StockCode", "Description", "Quantity",
     "InvoiceDate", "Price", "Customer ID", "Country", "SourceSheet"]
].head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom,Year 2009-2010
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom,Year 2009-2010
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom,Year 2009-2010
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom,Year 2009-2010
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom,Year 2009-2010
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,NaN,United Kingdom,Year 2009-2010
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,NaN,United Kingdom,Year 2009-2010
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom,Year 2009-2010
4538,489820,21133,invcd as 84879?,-720,2009-12-02 13:23:00,0.0,NaN,United Kingdom,Year 2009-2010
4566,489821,85049G,NaN,-240,2009-12-02 13:25:00,0.0,NaN,United Kingdom,Year 2009-2010


In [11]:
zero_price_sales = sales_df[sales_df["Price"] == 0]
negative_price_sales = sales_df[sales_df["Price"] < 0]

print("Zero price rows:", len(zero_price_sales))
print("Negative price rows:", len(negative_price_sales))

Zero price rows: 6191
Negative price rows: 5


In [12]:
negative_price_sales[
    ["Invoice", "StockCode", "Description", "Quantity",
     "InvoiceDate", "Price", "Customer ID", "Country", "SourceSheet"]
]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom,Year 2009-2010
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom,Year 2009-2010
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom,Year 2009-2010
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom,Year 2010-2011
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom,Year 2010-2011


In [13]:
print("Missing Customer IDs:",
      sales_df["Customer ID"].isna().sum())

print("Missing Descriptions:",
      sales_df["Description"].isna().sum())

Missing Customer IDs: 242125
Missing Descriptions: 4382


In [14]:
before = len(sales_df)

sales_df = sales_df[
    (sales_df["Quantity"] > 0) &
    (sales_df["Price"] > 0) &
    (sales_df["Description"].notna())
].copy()

after = len(sales_df)

print("Rows before:", before)
print("Rows after:", after)
print("Rows removed:", before - after)

Rows before: 1035805
Rows after: 1029609
Rows removed: 6196


In [15]:
print("Rows with missing Customer ID:",
      sales_df["Customer ID"].isna().sum())

print("Total sales rows:",
      len(sales_df))

Rows with missing Customer ID: 236000
Total sales rows: 1029609


In [16]:
customer_df = sales_df[
    sales_df["Customer ID"].notna()
].copy()

print("Customer analysis rows:", len(customer_df))
print("Rows excluded due to missing Customer ID:",
      len(sales_df) - len(customer_df))

Customer analysis rows: 793609
Rows excluded due to missing Customer ID: 236000


In [17]:
sales_df["Revenue"] = sales_df["Quantity"] * sales_df["Price"]
customer_df["Revenue"] = customer_df["Quantity"] * customer_df["Price"]

sales_df[
    ["Invoice", "Description", "Quantity", "Price", "Revenue"]
].head()

,Invoice,Description,Quantity,Price,Revenue
0,489434,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,6.95,83.4
1,489434,PINK CHERRY LIGHTS,12,6.75,81.0
2,489434,WHITE CHERRY LIGHTS,12,6.75,81.0
3,489434,"RECORD FRAME 7"" SINGLE SIZE",48,2.10,100.8
4,489434,STRAWBERRY CERAMIC TRINKET BOX,24,1.25,30.0


In [18]:
sales_df["InvoiceDate"] = pd.to_datetime(sales_df["InvoiceDate"])
customer_df["InvoiceDate"] = pd.to_datetime(customer_df["InvoiceDate"])

sales_df["Customer ID"] = sales_df["Customer ID"].astype("Int64")
customer_df["Customer ID"] = customer_df["Customer ID"].astype("Int64")

In [19]:
for data in [sales_df, customer_df]:
    data["Year"] = data["InvoiceDate"].dt.year
    data["Month"] = data["InvoiceDate"].dt.month
    data["MonthName"] = data["InvoiceDate"].dt.month_name()
    data["YearMonth"] = data["InvoiceDate"].dt.to_period("M").astype(str)
    data["DayOfWeek"] = data["InvoiceDate"].dt.day_name()
    data["Hour"] = data["InvoiceDate"].dt.hour

In [20]:
print("Sales rows:", len(sales_df))
print("Customer rows:", len(customer_df))

print("Duplicate rows:", sales_df.duplicated().sum())
print("Negative quantities:", (sales_df["Quantity"] <= 0).sum())
print("Zero/negative prices:", (sales_df["Price"] <= 0).sum())
print("Missing descriptions:", sales_df["Description"].isna().sum())
print("Missing Customer IDs:", sales_df["Customer ID"].isna().sum())

Sales rows: 1029609
Customer rows: 793609
Duplicate rows: 0
Negative quantities: 0
Zero/negative prices: 0
Missing descriptions: 0
Missing Customer IDs: 236000


In [21]:
sales_df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,IsCancelled,Revenue,Year,Month,MonthName,YearMonth,DayOfWeek,Hour
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,Year 2009-2010,False,83.4,2009,12,December,2009-12,Tuesday,7
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Year 2009-2010,False,81.0,2009,12,December,2009-12,Tuesday,7
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Year 2009-2010,False,81.0,2009,12,December,2009-12,Tuesday,7
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,Year 2009-2010,False,100.8,2009,12,December,2009-12,Tuesday,7
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,Year 2009-2010,False,30.0,2009,12,December,2009-12,Tuesday,7


In [22]:
processed_dir = project_root / "data" / "processed"

processed_dir.mkdir(parents=True, exist_ok=True)

print(processed_dir)

/Users/nithinsaijatoth/Desktop/UK_Retail_Sales_Customer_Analytics/data/processed


In [23]:
sales_output = processed_dir / "cleaned_sales.csv"

sales_df.to_csv(sales_output, index=False)

print("Saved:", sales_output)

Saved: /Users/nithinsaijatoth/Desktop/UK_Retail_Sales_Customer_Analytics/data/processed/cleaned_sales.csv


In [24]:
customer_output = processed_dir / "customer_sales.csv"

customer_df.to_csv(customer_output, index=False)

print("Saved:", customer_output)

Saved: /Users/nithinsaijatoth/Desktop/UK_Retail_Sales_Customer_Analytics/data/processed/customer_sales.csv


In [25]:
cancelled_output = processed_dir / "cancelled_transactions.csv"

cancelled_df.to_csv(cancelled_output, index=False)

print("Saved:", cancelled_output)

Saved: /Users/nithinsaijatoth/Desktop/UK_Retail_Sales_Customer_Analytics/data/processed/cancelled_transactions.csv


In [ ]:
mysql_export = sales_df[
    [
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "Price",
        "Customer ID",
        "Country",
        "IsCancelled",
        "Revenue"
    ]
].copy()

mysql_export["IsCancelled"] = mysql_export["IsCancelled"].astype(int)

mysql_export.to_csv(
    processed_dir / "cleaned_sales_mysql.csv",
    index=False
)

print("MySQL-ready file saved.")